# 15 — Blend3: A1 / MSC / CR (Continuum Removal)

**Strategy**: 3 preprocessing components, each trained with ExtraTrees,  
blended by simple average. No CV-based weight optimization (CV-LB anti-correlated).  
Purpose: binary search to isolate value of each component.

- **A1**: log10(1/R) -> SNV -> SG1(11,2,1)
- **MSC**: Multiplicative Scatter Correction -> SG1(11,2,1)
- **CR**: Continuum Removal (upper convex hull, reflectance R), div and sub versions

**LB calibration** (inverse correlation confirmed):  
CV=12.96->LB=21.20, CV=14.20->LB=19.87, CV=16.23->LB=18.35 (current best)

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupKFold

from src.utils import load_data, parse_spectra, get_groups, make_submission
from src.preprocessing import snv, msc, savitzky_golay

SEED = 42
ET_KW = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)
CLIP_T = 200.0

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

# ---- Metrics ----
def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan

# Sort wavenumber indices ascending (for convex hull)
WN_ORDER = np.argsort(wn)         # index into spectrum for ascending wn
WN_SORTED = wn[WN_ORDER]          # wn sorted ascending
WN_INV    = np.argsort(WN_ORDER)  # inverse: restore original order

print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'wn range: {wn.min():.0f} - {wn.max():.0f} cm^-1')
print(f'R range: {X_raw.min():.4f} - {X_raw.max():.4f}')
print(f'Folds: {len(SPLITS)}')

In [ ]:
# ==== A1: Absorbance (log10(1/R)) -> SNV -> SG1(11,2,1) ====
def preproc_A1(X_R):
    A = np.log10(1.0 / np.clip(X_R.astype(float), 1e-6, None))
    A = snv(A)
    return savitzky_golay(A, window_length=11, polyorder=2, deriv=1)

# ==== MSC: Multiplicative Scatter Correction -> SG1(11,2,1) ====
# ref must be fit on training data only (fold-internal / full-train)
def fit_msc_ref(X_R_tr):
    return X_R_tr.mean(axis=0)

def preproc_MSC(X_R, ref):
    Xm = msc(X_R.astype(float), reference=ref)
    return savitzky_golay(Xm, window_length=11, polyorder=2, deriv=1)

# ==== CR: Continuum Removal via Upper Convex Hull ====
# Per-spectrum operation — no fitting needed from training data
def _upper_hull_interp(wn_s, R_s):
    """Upper convex hull of (wn_s, R_s), both in ascending wn order.
    Returns continuum interpolated at all wn_s positions."""
    n = len(wn_s)
    hull = []  # indices into wn_s / R_s
    for i in range(n):
        while len(hull) >= 2:
            o, a = hull[-2], hull[-1]
            # Cross product (A-O) x (B-O); pop A if >= 0 (left-turn or collinear)
            cross = ((wn_s[a]-wn_s[o]) * (R_s[i]-R_s[o])
                     - (R_s[a]-R_s[o]) * (wn_s[i]-wn_s[o]))
            if cross >= 0:
                hull.pop()
            else:
                break
        hull.append(i)
    hull_wn = wn_s[hull]
    hull_R  = R_s[hull]
    return np.interp(wn_s, hull_wn, hull_R)

def preproc_CR(X_R):
    """Returns (CR_div, CR_sub) both shape (n, p) in original wn order."""
    X = X_R.astype(float)
    n, p = X.shape
    CR_div = np.empty((n, p))
    CR_sub = np.empty((n, p))
    for i in range(n):
        R_s = X[i][WN_ORDER]          # sort ascending by wn
        cont = _upper_hull_interp(WN_SORTED, R_s)
        cont = np.maximum(cont, 1e-8)
        CR_div[i] = (R_s / cont)[WN_INV]       # restore original order
        CR_sub[i] = (cont - R_s)[WN_INV]
    return CR_div, CR_sub

# Quick sanity check on a single spectrum
test_row = X_raw[0]
test_R_s = test_row[WN_ORDER]
test_cont = _upper_hull_interp(WN_SORTED, test_R_s)
print(f'CR sanity: R range=[{test_R_s.min():.3f},{test_R_s.max():.3f}]  '
      f'continuum range=[{test_cont.min():.3f},{test_cont.max():.3f}]')
print(f'CR_div range=[{(test_R_s/np.maximum(test_cont,1e-8)).min():.3f},'
      f'{(test_R_s/np.maximum(test_cont,1e-8)).max():.3f}]')
print('Preprocessing functions defined.')

## CV Health Check

**CV-LB anti-correlation is confirmed** — CV scores are NOT used for model selection.  
Purpose: verify prediction distributions are physically reasonable (mean ~40-50%, no extreme bias).  
All CV fits are fold-internal (MSC ref on training fold only).

In [ ]:
print('=== CV Health Check (RMSE_le170, fold-internal, health only) ===')
print('NOTE: Lower CV does NOT mean better LB.')

cv_results = {}

for comp_name in ['A1', 'MSC', 'CR_div', 'CR_sub']:
    fold_le = []
    oof_y, oof_p = [], []
    for fi, (tr, va) in enumerate(SPLITS):
        Xtr_R, Xva_R = X_raw[tr], X_raw[va]
        ytr, yva = y[tr], y[va]

        if comp_name == 'A1':
            Xtr = preproc_A1(Xtr_R)
            Xva = preproc_A1(Xva_R)
        elif comp_name == 'MSC':
            ref = fit_msc_ref(Xtr_R)
            Xtr = preproc_MSC(Xtr_R, ref)
            Xva = preproc_MSC(Xva_R, ref)
        elif comp_name == 'CR_div':
            Xtr, _ = preproc_CR(Xtr_R)
            Xva, _ = preproc_CR(Xva_R)
        elif comp_name == 'CR_sub':
            _, Xtr = preproc_CR(Xtr_R)
            _, Xva = preproc_CR(Xva_R)

        m = ExtraTreesRegressor(**ET_KW)
        m.fit(Xtr, ytr)
        pred = m.predict(Xva)
        fold_le.append(round(rmse_le(yva, pred), 2))
        oof_y.append(yva); oof_p.append(pred)

    oof_y_all = np.concatenate(oof_y)
    oof_p_all = np.concatenate(oof_p)
    mean_le = round(float(np.mean(fold_le)), 2)
    cv_results[comp_name] = {'folds': fold_le, 'mean': mean_le,
                             'oof_y': oof_y_all, 'oof_p': oof_p_all}
    print(f'  {comp_name:8s}: RMSE_le170={mean_le:.2f}%  folds={fold_le}')

print('CV health check done.')

## Test Predictions (fit on full training data)

In [ ]:
print('=== Generating test predictions (fit on full train) ===')

test_preds = {}

# --- A1 ---
Xtr_A1 = preproc_A1(X_raw)
Xte_A1 = preproc_A1(X_test_raw)
m_A1 = ExtraTreesRegressor(**ET_KW)
m_A1.fit(Xtr_A1, y)
test_preds['A1'] = np.clip(m_A1.predict(Xte_A1), 0, CLIP_T)
print(f'  A1:     min={test_preds["A1"].min():.1f}  '
      f'mean={test_preds["A1"].mean():.1f}  '
      f'max={test_preds["A1"].max():.1f}  '
      f'>170: {(test_preds["A1"]>170).sum()}')

# --- MSC ---
ref_full = fit_msc_ref(X_raw)  # full train mean
Xtr_MSC = preproc_MSC(X_raw, ref_full)
Xte_MSC = preproc_MSC(X_test_raw, ref_full)
m_MSC = ExtraTreesRegressor(**ET_KW)
m_MSC.fit(Xtr_MSC, y)
test_preds['MSC'] = np.clip(m_MSC.predict(Xte_MSC), 0, CLIP_T)
print(f'  MSC:    min={test_preds["MSC"].min():.1f}  '
      f'mean={test_preds["MSC"].mean():.1f}  '
      f'max={test_preds["MSC"].max():.1f}  '
      f'>170: {(test_preds["MSC"]>170).sum()}')

# --- CR (both versions from one pass) ---
print('  Computing CR convex hull (full train + test)...')
Xtr_CR_div, Xtr_CR_sub = preproc_CR(X_raw)
Xte_CR_div, Xte_CR_sub = preproc_CR(X_test_raw)

m_CR_div = ExtraTreesRegressor(**ET_KW)
m_CR_div.fit(Xtr_CR_div, y)
test_preds['CR_div'] = np.clip(m_CR_div.predict(Xte_CR_div), 0, CLIP_T)
print(f'  CR_div: min={test_preds["CR_div"].min():.1f}  '
      f'mean={test_preds["CR_div"].mean():.1f}  '
      f'max={test_preds["CR_div"].max():.1f}  '
      f'>170: {(test_preds["CR_div"]>170).sum()}')

m_CR_sub = ExtraTreesRegressor(**ET_KW)
m_CR_sub.fit(Xtr_CR_sub, y)
test_preds['CR_sub'] = np.clip(m_CR_sub.predict(Xte_CR_sub), 0, CLIP_T)
print(f'  CR_sub: min={test_preds["CR_sub"].min():.1f}  '
      f'mean={test_preds["CR_sub"].mean():.1f}  '
      f'max={test_preds["CR_sub"].max():.1f}  '
      f'>170: {(test_preds["CR_sub"]>170).sum()}')

# --- Blends ---
# Primary 3-component: A1 + MSC + CR_div
test_preds['blend3_div'] = np.clip(
    (test_preds['A1'] + test_preds['MSC'] + test_preds['CR_div']) / 3.0,
    0, CLIP_T)
# Alternate 3-component: A1 + MSC + CR_sub
test_preds['blend3_sub'] = np.clip(
    (test_preds['A1'] + test_preds['MSC'] + test_preds['CR_sub']) / 3.0,
    0, CLIP_T)
# A1+MSC only (to isolate CR contribution via subtraction)
test_preds['blend_A1_MSC'] = np.clip(
    (test_preds['A1'] + test_preds['MSC']) / 2.0,
    0, CLIP_T)

for key in ['blend3_div', 'blend3_sub', 'blend_A1_MSC']:
    p = test_preds[key]
    print(f'  {key:15s}: min={p.min():.1f}  mean={p.mean():.1f}  '
          f'max={p.max():.1f}  >170: {(p>170).sum()}')

print('Test predictions done.')

## Submission Files

In [ ]:
os.makedirs('../submissions', exist_ok=True)

SUB_MAP = [
    ('sub_A1',          'A1',          'A1: absorbance->SNV->SG1'),
    ('sub_MSC',         'MSC',         'MSC->SG1'),
    ('sub_CR_div',      'CR_div',      'CR division (R/continuum)'),
    ('sub_CR_sub',      'CR_sub',      'CR subtraction (continuum-R)'),
    ('sub_blend3_div',  'blend3_div',  'Blend3: A1+MSC+CR_div (equal weight)'),
    ('sub_blend3_sub',  'blend3_sub',  'Blend3: A1+MSC+CR_sub (equal weight)'),
    ('sub_blend_A1_MSC','blend_A1_MSC','Blend2: A1+MSC (no CR, for CR diff)'),
]

print('Submission files:')
print(f'  {"Name":<20s} {"CV%":>6s} {"min":>6s} {"mean":>6s} {"max":>6s} {"Description"}')
print('-' * 85)

for fname, key, desc in SUB_MAP:
    p = test_preds[key]
    cv_str = f'{cv_results[key]["mean"]:.2f}' if key in cv_results else 'blend'
    path = f'../submissions/{fname}.csv'
    make_submission(test_meta, p, path)
    print(f'  {fname:<20s} {cv_str:>6s}  {p.min():>6.1f} {p.mean():>6.1f} '
          f'{p.max():>6.1f}  {desc}')

print()
print('Recommended submission order (binary search logic):')
print('  1. sub_blend3_div  -- Does A1+MSC+CR blend beat current best (LB=18.35)?')
print('  2. sub_CR_div      -- Is CR alone a better component than SG1 variants?')
print('  3. sub_blend_A1_MSC-- A1+MSC without CR: isolate CRs contribution')
print('  4. sub_A1          -- A1 alone: absorbance derivative value')
print('  5. sub_MSC         -- MSC alone: scatter correction value')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Spectrum visualization
ax = axes[0, 0]
idx = 0
ax.plot(WN_SORTED, X_raw[idx][WN_ORDER], label='Raw R')
cont_ex = _upper_hull_interp(WN_SORTED, X_raw[idx][WN_ORDER])
ax.plot(WN_SORTED, cont_ex, 'r--', lw=2, label='Continuum (hull)')
ax.set_xlabel('Wavenumber (cm^-1)')
ax.set_title('Example: Raw R + Continuum (sample 0)')
ax.legend()
ax.invert_xaxis()

# CR_div
ax = axes[0, 1]
cr_div_ex = X_raw[idx][WN_ORDER] / np.maximum(cont_ex, 1e-8)
ax.plot(WN_SORTED, cr_div_ex)
ax.axhline(1.0, color='r', ls='--', lw=0.8)
ax.set_xlabel('Wavenumber (cm^-1)')
ax.set_title('CR_div = R / continuum (sample 0)')
ax.invert_xaxis()

# OOF scatter: A1 vs MSC
ax = axes[1, 0]
m_a1 = cv_results['A1']['oof_p'] <= 170
ax.scatter(cv_results['A1']['oof_y'][m_a1], cv_results['A1']['oof_p'][m_a1],
           s=5, alpha=0.3, label='A1')
ax.scatter(cv_results['MSC']['oof_y'][m_a1], cv_results['MSC']['oof_p'][m_a1],
           s=5, alpha=0.3, label='MSC')
ax.plot([0,170],[0,170],'k--',lw=0.8)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title('OOF scatter (y <= 170)')
ax.legend()

# Test prediction distributions
ax = axes[1, 1]
for key in ['A1', 'MSC', 'CR_div', 'CR_sub', 'blend3_div']:
    ax.hist(test_preds[key], bins=40, alpha=0.4, label=key)
ax.set_xlabel('Predicted moisture content (%)')
ax.set_title('Test prediction distributions')
ax.legend(fontsize=7)

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/nb15_blend3.png', dpi=110)
plt.close()
print('Saved: results/nb15_blend3.png')